In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 21 14:12:42 2020

@author: clara, viggy
"""

import numpy as np
import matplotlib.pyplot as plt
import copy
import statistics
import pandas as pd
import os
import re
import glob
from func_stepFrequency import get_gait, get_peaks, calc_step_width, calculate_step_stride_length, calc_interstep_all, get_step_lengths, get_frames, variability, asymmetry, intersect, calculate_stance_swing, calculate_percent_cycle, double_support_time, apply_lowpass_filter
import traceback

In [17]:
new_frames = pd.DataFrame()
error_subj = []
error_trace = []
error_idx = []

In [18]:
INDICES = ["trial_num", "test_type",
                      "Avg_LF_stance_time", 
                      "Avg_LF_swing_time",
                      "Avg_RF_stance_time",
                      "Avg_RF_swing_time",
                      "Variability_LF_stance_time",
                      "Variability_LF_swing_time",
                      "Variability_RF_stance_time",
                      "Variability_RF_swing_time",
                      "Asymmetry_Stance_time",
                      "Asymmetry_Swings_time",
                      "total_%_swing_time",
                      "total_%_stance_time",
                      "double_support_time",
                      "Avg_InterStepTime",
                      "Variability_InterStepTime",
                      "Frequency",
                      "Variance_COM_y",
                      "speed",
                      "Avg_stride_length",
                      "Variance_stride_length",
                      "Variability_stride_length",
                      "Asymmetry_stride_length",
                      "Avg_step_length",
                      "Variance_step_length",
                      "Variability_step_length",
                      "Avg_step_width",
                      "Variance_step_width",
                      "Variability_step_width",
                      "Avg_LF_max_height",
                      "Avg_RF_max_height",
                      "Variability_LF_max_height",
                      "Variability_RF_max_height",
                      "Asymmetry_F_max_height"]

## Load COM Data

In [ ]:
#filelocation = glob.glob(f'../data/gang/*.txt')
#filelocation = glob.glob(f'Patient_txt_PD_reup/*.txt')
#filelocation = glob.glob(f'Patient_txt_ataxia_reup/*.txt')
#filelocation = glob.glob(f'missing_txt_files_20250818/*.txt')
filelocation = glob.glob(f'all_participants_061826/*.txt')


all_results = pd.DataFrame()
counter = 0
PATTERN = r'-[2-9]-'

for files in filelocation:
    try:
        data = np.loadtxt(files, encoding = 'latin1') 
        #sub_name = [s[0:6] for s in os.path.basename(files).split()]
        file_strip = os.path.basename(files)
        sub_name = re.match(r'\d+', file_strip).group()  # leading digits: 101-001->101, 215147-001->215147 (handles 3- and 6-digit IDs)
        
        #search for multiple trials
        search = re.search(PATTERN, file_strip)
        if search != None: #match
            trial_num = search.group(0)[1]
        else: #no match
            trial_num = "1"
    
        subject = sub_name
        
        '''
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        '''
        
        if 'normalergang' in files.lower():
            test_type = 'NG'
            #sub_name += num
            #sub_name += "_NG"
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
        elif 'rückwärtsgang' in files.lower():
            test_type = 'RG'
            #sub_name += num
            #sub_name += '_RG'
            #lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']
        elif 'tandemgang' in files.lower():
            test_type = 'TG'
            #sub_name += num
            #sub_name += '_TG'
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']

        timeframes = data[:,1] #fractions of a second?
        time = timeframes / 60 #seconds or minutes?
        
        COM_x = data [:,2] #sagittal / anterior posterior
        COM_y = data[:,3] # right left
        COM_z = data[:,4] # vertical / up down
        # Subject 101 was exported with an extra Neck segment (38 cols vs the usual 32),
        # which shifts every position block after it by +3, so the feet land 3 columns later.
        # COM/Pelvis (first position block, cols 2-4) is unaffected. Detect by column count.
        foot_off = 3 if data.shape[1] == 38 else 0
        LeftFoot_x = data [:,11+foot_off]
        LeftFoot_y = data [:,12+foot_off]
        LeftFoot_z = data [:,13+foot_off]
        RightFoot_x = data [:,14+foot_off] #front-back in m
        RightFoot_y = data [:,15+foot_off] #left-right
        RightFoot_z = data [:,16+foot_off] #height

        # ---- Butterworth low-pass at construction (4th-order, 6 Hz, zero-phase) ----
        # Filter all position channels once here so every downstream consumer (get_peaks, the
        # stance/swing + DST path, and the x/y distance features: stride/step length, step
        # width, speed, COM_y variance) sees the same filtered signal. get_peaks runs with
        # filt=False (its default) to avoid double-filtering. Note: COM_x feeds gait-plate/lane
        # detection below, so that now also runs on filtered data (negligible on a slow path).
        COM_x = apply_lowpass_filter(COM_x)
        COM_y = apply_lowpass_filter(COM_y)
        COM_z = apply_lowpass_filter(COM_z)
        LeftFoot_x = apply_lowpass_filter(LeftFoot_x)
        LeftFoot_y = apply_lowpass_filter(LeftFoot_y)
        LeftFoot_z = apply_lowpass_filter(LeftFoot_z)
        RightFoot_x = apply_lowpass_filter(RightFoot_x)
        RightFoot_y = apply_lowpass_filter(RightFoot_y)
        RightFoot_z = apply_lowpass_filter(RightFoot_z)

        COM_z_time = np.array([time]+[COM_z]) #COM_z values with corresponding time
        COM_x_time = np.array([time]+[COM_x])
        COM_y_time = np.array([time]+[COM_y])

        
        ## Calculate Time at Gait Plate
        maxCOMx = max(COM_x)-1
        minCOMx = min(COM_x)+1


        COM_x_new = copy.copy(COM_x)
        COM_x_new[COM_x_new > maxCOMx] = 9.999
        COM_x_new[COM_x_new < minCOMx] = 9.999
        COM_x_new_time = np.array([time] + [COM_x_new])
        
        _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)
    
        com_data = {}
        lf_data = {}
        rf_data = {} 

        ## Build COM data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = COM_x_new_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = COM_y_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = COM_z_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            com_data[lane_name] = [x_data, y_data, z_data]

        ## Build Left/Right Foot data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = LeftFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = LeftFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = LeftFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            lf_data[lane_name] = [x_data, y_data, z_data]

            x_data = RightFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = RightFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = RightFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            rf_data[lane_name] = [x_data, y_data, z_data]

        
        var_all = []
        v_all = []

        Stride_length_all = [] #actually stride lengths
        Stride_length_all_L = [] 
        Stride_length_all_R = [] 

        step_width_all = []
        LF_height_sum = []
        RF_height_sum = []
        step_lengths = [] # step_lengths

        #swing & stances
        all_stances_L = []
        all_stances_R = []
        all_swings_L = []
        all_swings_R = []

        #percent swing and stances, currently calculated with all
        all_pc_st = []
        all_pc_sw = []
        
        #double support times for each lane in file
        dst_list = []

        for lane in lanes:
            #print(f'-------------------------{lane}-------------------------------')
            COM_xy_Lane_tp_new,  LF_xy_Lanetp_new, RF_xy_Lanetp_new = get_gait(lane, com_data, lf_data, rf_data, plot='off')

            var_lane = statistics.variance(COM_xy_Lane_tp_new[0,:])
            var_all.append(var_lane)
            
            v_lane = abs((COM_xy_Lane_tp_new[1,-1]-COM_xy_Lane_tp_new[1,0])/(com_data[lane][0][0,-1]-com_data[lane][0][0,0])) #(pos_end - pos_start) / (time_end - time_start)
            
            v_all.append(v_lane) #average of this is the speed
            
            peaks_COM, peaks_left, peaks_right, peaks_left_corrected, peaks_right_corrected, peaks_timeonly, peaks_heightonly, minima_RF, minima_LF = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
            
            #STRIDE LENGTH
            stride_length, stride_length_L, stride_length_R = calculate_step_stride_length(lane, com_data, lf_data, rf_data)
            Stride_length_all += list(stride_length) 
            Stride_length_all_L += list(stride_length_L)
            Stride_length_all_R += list(stride_length_R)
            
            #STEP LENGTH
            step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
            
            #STEP WIDTH
            step_width_all.append(calc_step_width(lane, com_data, lf_data, rf_data))

            LF_height_sum += list(peaks_left_corrected)
            RF_height_sum += list(peaks_right_corrected)
            
            #SWING AND STANCE TIME
            R, L, _, _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot="off") ##
            swing_R, stance_R, swing_L, stance_L, dst_base = calculate_stance_swing(lane, L, R, lf_data, rf_data, com_data)
            dst = double_support_time(dst_base)
            pc_sw, pc_st = calculate_percent_cycle(swing_R, stance_R, swing_L, stance_L)
            
            dst_list.append(dst)
            
            all_stances_L += list(stance_L)
            all_stances_R += list(stance_R)
            all_swings_L += list(swing_L)
            all_swings_R += list(swing_R)
            
            all_pc_st.append(pc_st)
            all_pc_sw.append(pc_sw)
            
            plt.close()
            
        ## STANCE/SWING ##

        all_stances_L = np.array(all_stances_L).flatten() #flatten to 1d
        all_stances_R = np.array(all_stances_R).flatten()
        all_swings_L = np.array(all_swings_L).flatten()
        all_swings_R = np.array(all_swings_R).flatten()

        # average swing & stance, each leg (4)
        stance_L_avg = np.mean(all_stances_L)
        stance_R_avg = np.mean(all_stances_R)
        swing_L_avg = np.mean(all_swings_L)
        swing_R_avg = np.mean(all_swings_R)

        # average % swing and % stance 
        pc_st_avg = np.mean(all_pc_st)
        pc_sw_avg = np.mean(all_pc_sw)

        # TOTAL stance and swing percentages --> effectively the same
        total = np.sum(all_stances_L) + np.sum(all_stances_R) + np.sum(all_swings_L) + np.sum(all_swings_R)
        total_pc_sw = (np.sum(all_swings_L) + np.sum(all_swings_R)) / total
        total_pc_st = (np.sum(all_stances_L) + np.sum(all_stances_R)) / total

        # variability, both legs (1)
        stances_L_variability = variability(all_stances_L)
        stances_R_variability = variability(all_stances_R)
        swings_L_variability = variability(all_swings_L)
        swings_R_variability = variability(all_swings_R)

        # asymmetry, between legs (1)
        stances_asymmetry = asymmetry(all_stances_L, all_stances_R)
        swings_asymmetry = asymmetry(all_swings_L, all_swings_R)
        
        #double support time
        dst_avg = np.mean(dst_list)

        #interstep & frequency
        InterStepTime_avg, InterStepTime_arr = calc_interstep_all(lanes, com_data, lf_data, rf_data)
        Variability_InterStepTime = variability(InterStepTime_arr)
        Frequency = 1/InterStepTime_avg

        #### Calculate Variance of COMy and Velocity/Speed

        ## Durchschnittsgeschwindigkeit:
        variance_COMy = np.mean(var_all)
        speed = np.mean(v_all)

        #STRIDE
        stride_length_avg = np.mean(Stride_length_all) # Average STRIDE Length, all step lengths from right and left foot...
        stride_length_var = statistics.variance(Stride_length_all) # Variance
        stride_length_variability = variability(Stride_length_all)
        stride_length_asymmetry = asymmetry(Stride_length_all_L, Stride_length_all_R)

        #STEP -- NEW
        ac_step_length_avg = np.mean(step_lengths)
        ac_step_length_variance = statistics.variance(step_lengths)
        ac_step_length_variability = variability(step_lengths)

        step_width_all = [item for sublist in step_width_all for item in sublist] # macht aus einem nested array ein 1D-array
        #print(step_width_all)
        ##plt.plot(step_width_all, 'x')
        step_width_avg = np.mean(step_width_all)
        var_step_width = statistics.variance(step_width_all)
        step_width_variability = variability(step_width_all)

        #### Calculate Foot Height at Maximum Elevation
        LF_height_avg = np.mean(LF_height_sum)
        RF_height_avg = np.mean(RF_height_sum)

        foot_height_asymmetry = asymmetry(LF_height_sum, RF_height_sum)
        LF_height_variability = variability(LF_height_sum)
        RF_height_variability = variability(RF_height_sum)
        
        df = pd.DataFrame([trial_num,
                    test_type, 
                   stance_L_avg,
                   swing_L_avg,
                   stance_R_avg,
                   swing_R_avg,
                   stances_L_variability,
                   swings_L_variability,
                   stances_R_variability,
                   swings_R_variability,
                   stances_asymmetry,
                   swings_asymmetry,
                   total_pc_sw,
                   total_pc_st,
                   dst_avg,
                   InterStepTime_avg,
                   Variability_InterStepTime,
                   Frequency,
                   variance_COMy,
                   speed,
                   stride_length_avg,
                   stride_length_var,
                   stride_length_variability,
                   stride_length_asymmetry,
                   ac_step_length_avg,
                   ac_step_length_variance,
                   ac_step_length_variability,
                   step_width_avg,
                   var_step_width,
                   step_width_variability,
                   LF_height_avg,
                   RF_height_avg,
                   LF_height_variability,
                   RF_height_variability,
                   foot_height_asymmetry], index= INDICES, columns = [sub_name])

        all_results = pd.concat([all_results, df.T], axis = 0)
        all_results.to_csv('all_participants_062126.csv')

        #df.to_csv(f'../export/{sub_name}.csv')
        #df.to_csv(f'out/{sub_name}.csv') 
        #print(f'Analysis done for subject: {sub_name}')
        #plt.close()
        counter += 1
        
    except Exception as e:
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        print(f'Exception occured: {e}')
        print(traceback.format_exc())
        error_subj.append(str(sub_name))
        error_trace.append(traceback.format_exc())
        error_idx.append(counter)
        counter += 1
        
        #printing plots only for debugging
        print("trying debugging plots")
        try:
            for lane in lanes:
                _ = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
                _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot = 'off') ##
        except Exception as e2:
            print(f'Exception occured: {e}')
            print(traceback.format_exc())

       
       
        

#print(results)
#results.to_csv(f'../export/results_walks.csv')
#results.to_excel(f'../export/results_walks.xlsx')


1, -1 4.0232116652687875 396
0, 0 -1 12.7 396
1, -1 0.8412994299035261 300
0, 0 -1 23.05 300
1, -1 4.07376763864 263
0, 0 -1 34.71666666666667 263
1, -1 0.8302474080759591 260
0, 0 -1 46.5 260
1, -1 3.9253797734718665 199
0, 0 -1 9.516666666666667 199
1, -1 0.8770143410134695 184
0, 0 -1 18.15 184
1, -1 3.923838986116004 169
0, 0 -1 24.5 169
1, -1 0.8812508466406954 166
0, 0 -1 31.283333333333335 166
1, -1 3.9261718283871434 163
0, 0 -1 37.233333333333334 163
1, -1 0.8892827071264033 163
0, 0 -1 43.63333333333333 163
1, -1 4.331885391170026 315
0, 0 -1 18.8 315
1, -1 1.0620615743882662 282
0, 0 -1 31.333333333333332 282
1, -1 4.342046272760201 281
0, 0 -1 42.25 281
1, -1 1.064199531855268 293
0, 0 -1 51.6 293
1, -1 4.332372125482119 279
0, 0 -1 61.93333333333333 279
1, -1 1.0620310363908212 284
0, 0 -1 71.75 284
1, -1 3.556310905386422 317
0, 0 -1 11.783333333333333 317
1, -1 1.0868968304658915 341
0, 0 -1 26.9 341
1, -1 3.5593879004729856 291
0, 0 -1 38.483333333333334 291
1, -1 1.086

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 1.2164420151483852 215
0, 0 -1 20.783333333333335 215
1, -1 4.0839070356449705 219
0, 0 -1 28.833333333333332 219
1, -1 1.212056991244131 224
0, 0 -1 36.03333333333333 224
1, -1 4.09033915601802 219
0, 0 -1 43.25 219
1, -1 1.2196541339883615 236
0, 0 -1 50.516666666666666 236
1, -1 3.958991983165187 326
0, 0 -1 9.5 326
double peak detected at end of signal, moving on...
1, -1 1.0673616954092267 369
0, 0 -1 21.616666666666667 369
1, -1 3.961554639528049 348
0, 0 -1 33.46666666666667 348
1, -1 1.0615260488874114 329
0, 0 -1 45.56666666666667 329
1, -1 4.058848906644428 250
0, 0 -1 12.716666666666667 250
1, -1 0.7189901367518172 239
0, 0 -1 20.283333333333335 239
1, -1 4.0463032625841056 239
0, 0 -1 28.05 239
1, -1 0.7127473333242841 258
0, 0 -1 35.96666666666667 258
1, -1 4.057046544383853 242
0, 0 -1 43.266666666666666 242
1, -1 0.7154848006739972 247
0, 0 -1 51.15 247
1, -1 4.470422935806704 244
0, 0 -1 10.283333333333333 244
1, -1 1.3037885544920258 238
0, 0 -1 18.41666666666666

/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py:357: RuntimeWarning: invalid value encountered in scalar divide
  dst = (np.sum(dst_times)/2) / len(dst_times)
/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py:377: RuntimeWarning: invalid value encountered in scalar divide
  percent_sw = (np.sum(sw_R) + np.sum(sw_L)) / sum_all
/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py:378: RuntimeWarning: invalid value encountered in scalar divide
  percent_st = (np.sum(st_R) + np.sum(st_L)) / sum_all


1, -1 4.009531543965325 343
0, 0 -1 14.733333333333333 343
1, -1 0.9297218961272541 309
0, 0 -1 25.616666666666667 309
1, -1 4.0183520209728885 283
0, 0 -1 37.35 283
1, -1 0.9181924226866403 308
0, 0 -1 46.93333333333333 308
1, -1 4.033142766723711 295
0, 0 -1 56.483333333333334 295
1, -1 0.9170600377743678 278
0, 0 -1 65.55 278
1, -1 4.005802938427814 259
0, 0 -1 12.016666666666667 259
1, -1 0.5875356021698718 220
0, 0 -1 20.6 220
1, -1 3.996825829170194 200
0, 0 -1 29.266666666666666 200
1, -1 0.5821963696659705 222
0, 0 -1 36.68333333333333 222
1, -1 3.9942327801513224 223
0, 0 -1 44.05 223
1, -1 0.582996840833189 243
0, 0 -1 52.68333333333333 243
1, -1 3.7950052241950343 571
0, 0 -1 26.666666666666668 571
1, -1 0.9166255687193716 776
0, 0 -1 57.15 776
double peak detected at end of signal, moving on...
1, -1 3.765180056993083 665
0, 0 -1 86.25 665
1, -1 0.9295585375212022 701
0, 0 -1 110.38333333333334 701
1, -1 0.8318880870511585 148
0, 0 -1 10.016666666666667 148
1, -1 3.69078750

...

1, -1 0.8587292658104366 301
0, 0 -1 26.833333333333332 301
1, -1 3.833970493156469 242
0, 0 -1 37.38333333333333 242
1, -1 0.8616471432197872 241
0, 0 -1 47.56666666666667 241
1, -1 3.8543825657009454 366
0, 0 -1 17.983333333333334 366
1, -1 1.1039667686661976 321
0, 0 -1 29.9 321
1, -1 3.8578511699495923 289
0, 0 -1 40.21666666666667 289
1, -1 1.0959160880947514 303
0, 0 -1 51.53333333333333 303
1, -1 1.0744890684430946 181
0, 0 -1 7.55 181
double peak detected at end of signal, moving on...
1, -1 4.019562298414146 674
0, 0 -1 23.1 674
double peak detected at end of signal, moving on...
1, -1 1.0837529677616857 483
0, 0 -1 48.266666666666666 483
double peak detected at end of signal, moving on...
1, -1 4.016253621952779 452
0, 0 -1 67.55 452
double peak detected at end of signal, moving on...
1, -1 3.632299323334265 454
0, 0 -1 19.75 454
1, -1 0.7011848796748451 472
0, 0 -1 37.416666666666664 472
1, -1 3.6200397795094355 385
0, 0 -1 51.916666666666664 385
1, -1 0.7163257534675543 391

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


file_strip 110-2-001-normalergang_Seg_RightFoot_LeftFoot_RightForeArm_LeftForeArm_Pelvis_Data_position_acceleration.txt
sub_name 110
trial_num 2
Starting analysis for subject: 110
---------index 232---------
Exception occured: index 0 is out of bounds for axis 0 with size 0
Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs5m8h0000gn/T/ipykernel_8818/225952812.py", line 236, in <module>
    step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py", line 482, in get_step_lengths
    if np.abs(left_steps[0] - center) > np.abs(right_steps[0] - center): #left first, add first two steps
              ~~~~~~~~~~^^^
IndexError: index 0 is out of bounds for axis 0 with size 0

trying debugging plots
1, -1 3.601778106211059 256
0, 0 -1 15

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


file_strip 109-2-001-normalergang_Seg_RightFoot_LeftFoot_RightForeArm_LeftForeArm_Pelvis_Data_position_acceleration.txt
sub_name 109
trial_num 2
Starting analysis for subject: 109
---------index 236---------
Exception occured: index 0 is out of bounds for axis 0 with size 0
Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs5m8h0000gn/T/ipykernel_8818/225952812.py", line 236, in <module>
    step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py", line 482, in get_step_lengths
    if np.abs(left_steps[0] - center) > np.abs(right_steps[0] - center): #left first, add first two steps
              ~~~~~~~~~~^^^
IndexError: index 0 is out of bounds for axis 0 with size 0

trying debugging plots
1, -1 3.933986566630126 388
0, 0 -1 17

In [9]:
for i in all_results["double_support_time"]:
    print(i)

0.0404021644566128


In [5]:
all_results.to_csv('out/all_results_dst.csv')


In [8]:
len(error_subj)

28

In [6]:
demos = pd.read_csv("Demographics_PD.csv")

In [8]:
#subset to get rid of
demos_sub = demos.iloc[0:28, ]

demos_sub["ID"] = demos_sub["ID"].astype(int)
demos_sub["ID"] = demos_sub["ID"].astype(str)

#replace NV values with as control
demos_sub["Control"][14:16] = pd.Series(["1","1"])

#create mapping from control to subject ID
map = demos_sub[["ID", "Control"]]
map.set_index('ID', inplace=True)

mapping = map["Control"].to_dict()


demos_sub["Control"] = demos_sub["ID"].map(mapping)
all_results["ID"] = all_results.index
all_results["Control"] = all_results["ID"].map(mapping) #create Control column

/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(int)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(str)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [9]:
all_results.to_csv("out/all_results_dst_ctrl.csv")